# Honey Bee Image Classification

A complete computer-vision workflow for distinguishing honey bees (*Apis*) from bumble bees (*Bombus*). The notebook covers data validation, stratified splitting, transfer learning, evaluation and error analysis.


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
DATA_ROOT = Path("..")
IMAGE_DIR = DATA_ROOT / "images"
LABEL_FILE = DATA_ROOT / "datasets" / "labels.csv"
IMG_SIZE = (200, 200)
BATCH_SIZE = 32


## 1. Load and validate metadata


In [ ]:
labels = pd.read_csv(LABEL_FILE, index_col=0)
labels = labels.dropna(subset=["genus"]).copy()
labels["genus"] = labels["genus"].astype(int)
labels["image_path"] = [str(IMAGE_DIR / f"{idx}.jpg") for idx in labels.index]
labels = labels[labels["image_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)
print(labels["genus"].value_counts().rename(index={0: "Apis", 1: "Bombus"}))
print(f"Usable images: {len(labels):,}")


In [ ]:
sample = labels.sample(min(8, len(labels)), random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    image = tf.keras.utils.load_img(row["image_path"], target_size=IMG_SIZE)
    ax.imshow(image)
    ax.set_title("Bombus" if row["genus"] else "Apis")
    ax.axis("off")
plt.tight_layout()


## 2. Create reproducible train, validation and test splits


In [ ]:
train_df, test_df = train_test_split(labels, test_size=0.20, stratify=labels["genus"], random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.20, stratify=train_df["genus"], random_state=SEED)
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})


In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image, tf.cast(label, tf.float32)

def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame["image_path"].values, frame["genus"].values))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)


## 3. Build a transfer-learning model

MobileNetV2 provides a compact pretrained feature extractor. Augmentation is applied only during training.


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
])
base = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
model.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.3),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)


## 4. Evaluate and inspect errors


In [ ]:
history_df = pd.DataFrame(history.history)
history_df[["loss", "val_loss"]].plot(title="Training and validation loss")
plt.show()

y_probability = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_probability >= 0.5).astype(int)
y_true = test_df["genus"].to_numpy()
print(classification_report(y_true, y_pred, target_names=["Apis", "Bombus"]))
print(f"ROC-AUC: {roc_auc_score(y_true, y_probability):.3f}")
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Apis", "Bombus"], yticklabels=["Apis", "Bombus"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()


In [ ]:
errors = test_df.assign(prediction=y_pred, probability=y_probability)
errors = errors[errors["genus"] != errors["prediction"]].head(8)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, row) in zip(axes.flat, errors.iterrows()):
    ax.imshow(tf.keras.utils.load_img(row["image_path"], target_size=IMG_SIZE))
    ax.set_title(f"actual={row['genus']}, pred={row['prediction']}")
    ax.axis("off")
plt.tight_layout()


## Limitations

The dataset is comparatively small and may contain background or collection biases. Performance should therefore be validated on images from independent sources before real-world use.
